In [0]:
import pandas as pd
from datetime import datetime
import requests

In [0]:
tmdb_key = dbutils.secrets.get(scope='tmdb_keys', key='tmdb_key')

In [0]:
#Trazendo os ids de filmes já presentes no banco
df_gold = spark.read.table('workspace.gold.recomendation_db_gold')
ids_df = df_gold.toPandas()['id']
ids_df = pd.DataFrame(ids_df)

In [0]:
print(datetime.now().year)
year_now =  datetime.now().year

####Trazendo os filmes do **ano atual** e do **ano passado** com mais de 100 avaliações. Assim mantemos a lista atualizada com cargas semanais de novos filmes relevantes.####

In [0]:
url = "https://api.themoviedb.org/3/discover/movie"

headers = {
    "accept": "application/json",
    "Authorization": tmdb_key
}

movies = []

#Dew ano em ano
for year in range(year_now-1, year_now+1):

    #De pagina em pagina
    for page in range(1, 501):
        params = {
            "primary_release_year": year,
            "vote_count.gte": 100,
            "sort_by": "popularity.desc",
            "page": page,
        }

        #
        response = requests.get(url, headers= headers, params=params)
        data = response.json()

        #para quando nao tem mais paginas naquele ano
        if len(data['results']) == 0:
            break
        
        #adiciona ao array
        for movie in data['results']:
            movies.append({
                "id": movie["id"],
                 "title": movie["title"],
                "overview": movie["overview"],
                "genre_ids": movie["genre_ids"],
                })

df = pd.DataFrame(movies)

In [0]:
#Limpeza
df = (df[df['overview'] != ""])
df = df.drop_duplicates(subset='id')
df = df[df["genre_ids"].apply(len) > 0]

In [0]:
headers = {
    "accept": "application/json",
    "Authorization": tmdb_key
}

keywords_list = []

session = requests.Session()

for movie_id in df['id']:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    response_keywords = session.get(url, headers=headers)
    movie_keyword = response_keywords.json()

    keywords = []

    for keyword in movie_keyword['keywords']:
        keywords.append(keyword['name'])

    keywords_list.append(" ".join(keywords))

df['keywords'] = keywords_list

In [0]:
genres = {
      28: "Action",
      12: "Abenteuer",
      16: "Animation",
      35: "Komödie",
      80: "Krimi",
      99: "Dokumentarfilm",
      18: "Drama",
      10751 :"Familie",
      14: "Fantasy",
      36: "Historie",
      27: "Horror",
      10402: "Musik",
      9648: "Mystery",
      10749: "Liebesfilm",
      878 : "Science Fiction",
      10770: "TV-Film",
      53: "Thriller",
      10752: "Kriegsfilm",
      37: "Western",
  }

df["genres_text"] = df["genre_ids"].apply(
    lambda ids: " ".join([genres.get(i, "") for i in ids])
)

df = df.drop('genre_ids', axis=1)

In [0]:
#Criando a coluna 'content' onde ficarão todas informações de cada filme
df['content'] = df['overview'] +" "+ df['keywords']*2 +" "+ df['genres_text']*3

In [0]:
df = df.drop(columns= ["keywords", "overview", "genres_text"])

In [0]:
#Atribuindo a um novo DF apenas os filmes novos
df_new = df[~df['id'].isin(ids_df['id'])]

if df_new.shape[0] > 0:
    spark_df = spark.createDataFrame(df_new)
    spark_df.write.mode('append').saveAsTable('workspace.gold.recomendation_db_gold') #Salvando
    print(f'{df_new.shape[0]} novos filmes adicionados.')
else:
    print('Não há atualizações.')